#### Loading packages

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

#### Setting root directory path

In [ ]:
ROOT = r'C:\Users\PC_DS_ECON_5\Desktop\data-analytics-python'
 

#### Loading data

We need the `pyarrow` package to import a `.parquet` datafile.

In [ ]:
hotels_europe_restr_df = pd.read_parquet(ROOT + '/data/hotels_europe_restr_df.parquet')
nlsy97_income_hours_df = pd.read_parquet(ROOT + '/data/nlsy97_income_hours_df.parquet')

#### Histogram, frequencies, relative frequencies

Let us restrict our attention to hotel `price`s (for hotels with 3-4 stars) in Berlin in November 2017.

In [ ]:
df0 = hotels_europe_restr_df.copy() # create a copy
df0 = (
    df0.loc[(df0['city'] == 'Berlin') 
            & (df0['yearmonth'] == '2017-11-01') # November 2017
            & df0['price'].notna() # Non-missing prices
            & (df0['stars'] >= 3) # at 3-star hotels
            & (df0['stars'] <= 4), # at most 4-star hotels
            ['hotel_id', 'price']] # keep columns/variables (hotel_id, price)
        .reset_index(drop=True) #
)
df0

Let us look at `frequencies` for hotel `price`s for our sample.

In [ ]:
df = df0.copy()
df = (
    df.value_counts(subset=['price']) # counting the number of occurrences for each value of price
    .reset_index() # formatting -> data frame two columns: values of the variable (price) and count 
    .sort_values(by='price', ascending=True) # sorting the rows according to price (ascending)
    .reset_index(drop=True) # formatting 
    .copy() # this might be unnecessary; but if unsure, it is general safer (for small datasets it is fine)
)
df

Let us visualise the frequencies as a `histogram`.

In [ ]:
fig, ax = plt.subplots(figsize=(8,4)) # figure with single axes, 8-inches width, 4-inches height

ax.bar(x=df['price'], height=df['count'], width=5)
ax.set_xlabel('Price')
ax.set_ylabel('Frequency')
# ax.yaxis.set_ticks(ticks=range(df['count'].max() + 1))

Let us now look at the `relative frequencies` of prices for our sample.

In [ ]:
# same as for frequencies
# except for: normalize=True

df = df0.copy()
df = (
    df
    .value_counts(subset=['price'], normalize=True) # relative frequencies for each value of price
    .reset_index() # dataframe with two columns
    .sort_values(by='price') # sorting by price
    .reset_index(drop=True) # formatting
    .copy()
)
df

Let us plot the corresponding histogram with `relative frequencies`.

In [ ]:
# essentially the same as for relative frequencies
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(x=df['price'], height=df['proportion'], width=5)
ax.set_xlabel('Price')
ax.set_ylabel('Relative frequency')

Let us display the shares/proportions as percentages.

In [ ]:
from matplotlib.ticker import PercentFormatter


fig, ax = plt.subplots()
ax.bar(x=df['price'], height=df['proportion'], width=2)
ax.set_xlabel('Price')
ax.set_ylabel('Relative frequency')

# for the vertical axis display labels as / convert labels to percentages (express them in percentage points)
# 0.04 --> 4%
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0)) # xmax=1 --> 1 = 100%


#### Relative frequencies and probabilities; random variables 

Let us take the original sample that we have here (hotel prices for Berlin Nov 2017 3-4 star hotels)

Thought experiment: 
 - take the values and corresponding relative frequencies
 - relative frequencies are probabilities
 - context: randomly drawing observations from this original sample 
 --> Probability distribution
 --> Random variable


A set of values together with their corresponding relative frequencies already defines an empirical probability distribution. (There are other ways to define probability distributions.)

The easiest way to illustrate this idea is through random sampling.

When drawing a value randomly from our sample of hotel prices, the probability of drawing a particular price corresponds to its relative frequency.

For example, we can see that the value $103$ occurs in $\approx 4.4\%$ of the cases.

In [ ]:
# objective: most frequently occurring value and its relative frequency

# display the first row of price-relative frequency dataframe 
# in descending order by relative frequency

df.sort_values(by='proportion', ascending=False).head(2)

1. Repeatedly (and independently) drawing observations from the original sample

2. Progressively increasing the sample size

3. Each time computing the relative frequency in the random sample of `price = 103`

4. As the sample size increases --> relative frequency in the random sample converges to approx. 4.4% (probability associated with `price = 103`) 


If we repeatedly draw random samples from the empirical probability distribution defined by the original sample, then the relative frequency of the value $103$ should converge to approximately $4.4\%$ as the sample size increases.

$ \text{relative frequency}\ \longrightarrow\ \text{probability} \ \ \ \ \text{as sample size increases.}$

This idea is also closely related to the `Law of Large Numbers`.

It is also the basis of the `frequentist interpretation of probability`.

By carrying out random draws from our artificial distribution of `price`, we are introducing a `random variable`.

Here we have a `discrete random variable` (as opposed to a `continuous r. v.`), because the random variable can only take a countable set of possible values, each associated with a probability of occurrence.

For a `discrete random variable`, the theoretical/population counterpart of empirical/sample relative frequencies is the `probability mass`. The `probability mass` associated with a value is the probability that the random variable takes this value. 

The `distribution` of a `discrete random variable` is defined by the correspondence between each possible value and the associated `probability mass`. 

Using this terminology, when drawing random samples for our random variable, the probability mass associated with the value $103$ is approximately $4.4\%$.

Let us progressively increase the sample size to one million independent random draws and calculate the relative frequency for the value $103$ for each sample size.

In [ ]:
# implementation 
# relative frequencies in the random sample converge to probability (relative frequency in the orginal sample)

rng = np.random.default_rng(seed=42) # random number generator rng using numpy.random

sample_size_max = 1000000 # 1 million random draws
sample_sizes = range(1,sample_size_max+1) # 1, 2, 3, 4, ...., 1 million (list of numbers)

# initializing the data frame for the random draws: 
# two columns: sample size; relative frequency of 103 up to that point
rdf = pd.DataFrame({'sample_size': sample_sizes,
                    'relative_frequency': np.zeros(shape=len(sample_sizes))})

# carrying out the random draws
sample_all = rng.choice(
    a=df['price'], # values
    p=df['proportion'], # corresponding probabilities
    size=sample_size_max, # sample size
    replace=True # with replacement: probability of drawing a specific value is always the same --> draws are independent
    )

rdf['relative_frequency'] =  (
    (sample_all == 103) # True(1)/False(0) values for each draw True(1) if draw=103 and False(0) otherwise
    .cumsum() # cumulative sum: counting Trues(1) from the first raw up to the specific row (frequency)
    / rdf['sample_size'] # (dividing by sample size) 
)

rdf['value_of_random_draw'] = sample_all

rdf

Let us visualise the convergence.

In [ ]:
# relative frequency of price=103 in the original sample
probability_mass = df.loc[df['price'] == 103, 'proportion'].iloc[0]

fig, ax = plt.subplots()

# horizontal x axis: sample size
# vertival y axis: relative frequency of price=103 in the random sample up to the particular draw
ax.plot(rdf['sample_size'], rdf['relative_frequency']) 

# add a horizontal line y = 0.44 (4.4%) 
ax.axhline(y=probability_mass, linestyle='--', color='red')

# to avoid scientific notation on the x-axis (to have 1 000 000 instead of 10^6) 
ax.ticklabel_format(axis='x', style='plain')

ax.set_xlabel('Sample size')
ax.set_ylabel('Relative frequency of 103')


#### Summary statistics for quantitative variables

##### Mean

The most common measure of central tendency is the `mean` (or the `arithmetic mean`).

For $N$ non-missing values of $x$, the mean is the sum of all observed values divided by the number of non-missing observations.

$$ 
\bar{x} = \frac{1}{N} \displaystyle \sum_{i=1}^{N} x_i 
$$

In [ ]:
# implementing the formula
print(df0['price'].sum()/df0['price'].dropna().shape[0])

print(df0['price'].mean())

Another way to think about the mean is that it is the relative-frequency-weighted sum of all unique values.  

$$ 
\bar{x} = \displaystyle \sum_{v \in \mathcal{V}_x} v \cdot \underbrace{\frac{n_x(v)}{N}}_{\mathclap{\substack{\text{relative frequency}}}} 
$$

where $ \mathcal{V}_x $ is the set of unique values of variable $x$ and $n_x(v)$ is the frequency of value $v$ for variable $x$. (To be pedantic: $ N = \sum_{v \in \mathcal{V}_x} n_x(v) $)

In [ ]:
# using data frame with relative frequencies
# multiply each unique value by the corresponding relative frequency, take the sum
(df['price'] * df['proportion']).sum()

##### Expected value and Law of Large Numbers

As we have seen above:
 - For observed data, we have `variables`, like `price` here; for theoretical distributions in the context of random sampling, we have `random variables`.
 - For observed data, possible values of variables are associated with `relative frequencies` of occurrence; for distributions of random variables, possible values of random variables are associated with `probabilities` of occurrence.

In line with this correspondence, 
 - For observed data, each variable has a (sample) `mean`; in the context of distributions of random variables, each random variable has an `expected value` (or theoretical/population mean).   
 - For observed data, the (sample) mean is the `relative-frequency-weighted sum of all unique values`. Analogously, for a discrete random variable, the expected value is the `probability-weighted sum of all possible values`.

$$
\mathbb{E}[X] = \displaystyle \sum_{v \in \mathcal{V}_X} v \cdot \underbrace{\mathbb{P}[X = v]}_{\mathclap{\substack{\text{probability mass}}}}
$$    

where $ \mathcal{V}_X $ is the set of possible values of the random variable $X$.


According to the Law of Large Numbers:

$ \text{sample mean}\ \longrightarrow\ \text{expected value} \ \ \ \ \text{as sample size increases.}$

Let us illustrate the LLN.

Let us use again our original sample of prices as an artificial distribution and carry out independent random draws from it.

Let us progressively increase the sample size to one million and calculate the sample mean for each sample size.

In [ ]:
rng = np.random.default_rng(seed=42)

sample_size_max = 1000000
sample_sizes = range(1,sample_size_max+1)
rdf = pd.DataFrame({'sample_size': sample_sizes,
                    'sample_mean': np.zeros(shape=len(sample_sizes))})
sample_all = rng.choice(
    a=df['price'],
    p=df['proportion'],
    size=sample_size_max,
    replace=True
    )

rdf['sample_mean'] =  sample_all.cumsum() / rdf['sample_size']
rdf

Let us visualize the convergence.


In [ ]:
expected_value = (df['price'] * df['proportion']).sum()

fig, ax = plt.subplots()

ax.plot(rdf['sample_size'], rdf['sample_mean'])
ax.axhline(y=expected_value, linestyle='--', color='red')
ax.ticklabel_format(axis='x', style='plain')
ax.set_xlabel('Sample size')
ax.set_ylabel('Sample mean')

To sum up:

| Empirical / Sample            | Theoretical / Population |
| ----------------------------- | ------------------------ |
| Relative frequency            | Probability (mass)       |
| (Sample) mean                 | Expected value           |


##### Variance and standard deviation

One of the most common measures of dispersion is the `variance`.

In the context of observed data, the variance is simply the `mean of squared deviations from the mean`:

$$ 
Var_x = \frac{1}{N} \displaystyle \sum_{i=1}^{N} (x_i - \bar{x})^2 
$$

In the context of random variables, the variance of a random variable is the `expected value of squared deviations from the expected value`:

$$ 
Var[X] = \mathbb{E}[(X - \mathbb{E}[X])^2] 
$$



In [ ]:
print(((df0['price'] - df0['price'].mean()) ** 2).mean())
print(df0['price'].var(ddof=0))

What is the issue with the units of this quantity?

The units are also squared --> 4174 dollars squared --> difficult to interpret

To obtain a measure of dispersion that has the same units as the original variable, we can simply take the square root of the variance and obtain the `standard deviation`.

$$
SD[X] = \sqrt{Var[X]}
$$

The advantage of the standard deviation is that it is expressed in the same units as the original variable. For example, if `price` is measured in `dollars`, then the `standard deviation` of `price` is also measured in `dollars`.

In [ ]:
print(df0['price'].std(ddof=0))
print(df0['price'].var(ddof=0) ** 0.5)


##### Standardisation or z-score

One can standardise a (random) variable by subtracting its mean (or expected value) and dividing by its standard deviation:

$$
z_i = \frac{x_i - \bar{x}}{SD_x}  \ \ \ \text{ or } \ \ \ Z = \frac{X - \mathbb{E}[X]}{SD[X]}
$$

Standardisation transforms a variable into a unitless measure of how far an observation lies from the mean, measured in standard deviations.

This makes observations easier to compare across different variables and different units of measurement.

Let us plot the relative frequencies by standardised score for our original sample.

In [ ]:
avg = df0['price'].mean() 
sd = df0['price'].std(ddof=0)

df['price_z_score'] = (df['price'] - avg)/sd

fig, ax = plt.subplots(figsize=(8,4))
ax.bar(x=df['price_z_score'], height=df['proportion'], width=0.1)
ax.set_xlabel('Price z-score')
ax.set_ylabel('Relative frequency')

Let us calculate the proportion of observations more than 2 standard deviations away from the mean.

In [ ]:
df.loc[df['price_z_score'].abs() > 2,'proportion'].sum()

#### Cumulative relative frequencies and probabilities; quantiles

##### Cumulative relative frequency plot

Let us compute `cumulative relative frequencies` for our original sample of hotel prices.

The cumulative relative frequency associated with a value `r` is the share of observations whose value is less than or equal to `r`:

$$ 
F_x(r) = \displaystyle \sum_{v \leq r} \underbrace{\frac{n_x(v)}{N}}_{\mathclap{\substack{\text{relative frequency}}}} 
$$

where $n_x(v)$ is the frequency of value $v$. 

Put differently, for each value, let us plot the share of observations that are less than or equal to it.

In [ ]:
df

In [ ]:
# df is table of relative relative frequencies
df = df.sort_values(by='price', ascending=True) # sort data in ascending order in terms of price
# cumulative relative frequency = cumulative sum of relative frequencies
df['cumulative_proportion'] = df['proportion'].cumsum() 
df

Let us first visualise the cumulative relative frequencies with a step function. 

In [ ]:
fig, ax = plt.subplots()
# step function
ax.step(x=df['price'], y=df['cumulative_proportion'], where='post')

ax.set_xlabel('Price')
ax.set_ylabel('Cumulative relative frequency')
ax.set_ylim(bottom=0, top=1) # cumulative relative frequencies should be between 0 and 1

Matplotlib also provides a built-in `ecdf` (empirical cumulative distribution function) plot type for the raw data.

In [ ]:
# using the original sample
# no need to calculate the (cumulative) relative frequencies by hand 
df2 = df0.copy()
fig, ax = plt.subplots()
# ecdf = Empirical Cumulative Distribution Function
ax.ecdf(x=df2['price'])
ax.set_xlabel('Price')
ax.set_ylabel('Cumulative relative frequency')

##### Cumulative distribution function

We have seen earlier that:

 - For observed data, we have `variables`, like `price` here; for theoretical distributions in the context of random sampling, we have `random variables`.
 - For observed data, possible values of variables are associated with `relative frequencies` of occurrence; for distributions of random variables, possible values of random variables are associated with `probabilities` of occurrence.

In line with this correspondence,

 - For observed data, one can define cumulative relative frequencies as the `sum of relative frequencies associated with values less than or equal to a given value`;
 - In the context of random variables, each random variable has a `cumulative distribution function` (or `CDF`) defined as the `probability that the random variable takes a value less than or equal to a given value`:

$$
F_X(v) = \mathbb{P}[X \leq v].
$$

To sum up:

| Empirical / Sample            | Theoretical / Population      |
| ----------------------------- | ------------------------------|
| Relative frequency            | Probability (mass)            |
| (Sample) mean                 | Expected value                |
| Cumulative relative frequency | Cumulative probability / CDF  |


##### Quantiles

Let us now plot the quantile function, the "inverse" of the cumulative relative frequency function.

Concretely, let us plot for each value `p` between `0` and `1`, the lowest value of `price` such that the corresponding cumulative relative frequency of `price`s is at least `p`:

$$
Q(p) = \inf\{ r : p \leq F(r) \}    
$$

where $F_x(r)$ is the cumulative relative frequency of price $r$.

Put slightly differently, the `p`-quantile ist the lowest value `r` such that at least `p`-proportion of all observed values for `price` fall below `r`.

According to this definition, the `p=0.5`-quantile is conceptually the same as the `median` but it may take on a slightly different value when the number of observations is even.

In [ ]:
fig, ax = plt.subplots()
# quantile 
ax.step(x=df['cumulative_proportion'], y=df['price'], where='pre')
ax.set_xlabel('Proportion (cumulative relative frequency)')
ax.set_ylabel('Price-Quantile')
ax.set_xlim(left=0, right=1) # proportions should be between 0 and 1

**Names for typical quantiles**

- median: $Q(0.5)$  
- quartiles: $Q(0.25)$, $Q(0.5)$, $Q(0.75)$ 
- quintiles: $Q(0.2)$, $Q(0.4)$, $Q(0.6)$, $Q(0.8)$
- deciles: $Q(0.1)$, $Q(0.2)$, $Q(0.3)$, $Q(0.4)$, $Q(0.5)$, $Q(0.6)$, $Q(0.7)$, $Q(0.8)$, $Q(0.9)$
- percentiles: $Q(0.01)$, $Q(0.02)$, $Q(0.03)$, $...$, $Q(0.97)$, $Q(0.98)$, $Q(0.99)$



Let us visualise three common quantiles using the quantile function.

In [ ]:
# 0.25-quantile (1st/lower quartile), 0.5-quantile (median), 0.75-quantile (3rd/upper quartile) 
cum_freqs = [0.25, 0.5, 0.75]

# to each cumulative proportion p assign the corresponding quantile Q(p)
quantiles = {p: df2['price'].quantile(p) for p in cum_freqs} # {} --> dictionary

fig, ax = plt.subplots(figsize = (6,6))
# empirical quantile function as a step function
ax.step(x=df['cumulative_proportion'], y=df['price'], where='pre')

ax.set_xlabel('Proportion (cumulative relative frequency)')
ax.set_ylabel('Price-Quantile (log scale)')
ax.set_xlim(left=0, right=1)

ax.set_ylim(bottom=df2['price'].min(), top=df2['price'].max())

# for each cumulative proportion in [0.25, 0.5, 0.75]
for p in cum_freqs: # for-loop
    # indicate the corresponding quantiles with the help of vertical and horizontal lines
    ax.plot([0,p], [quantiles[p], quantiles[p]], linestyle='--', color='red', linewidth=1)
    ax.plot([p,p], [df2['price'].min(), quantiles[p]], linestyle='--', color='red', linewidth=1)

# log-scale because the distribution of prices is right-skewed
# --> there is long right tail
ax.set_yscale(value='log')

# labelling the axis x
ax.set_xticks(
    [0.25, 0.50, 0.75],
    labels=['25%', '50%', '75%']
)
# labelling the axis y
ax.set_yticks(
    [quantiles[0.25], quantiles[0.50], quantiles[0.75]],
    labels=[f'Lower quartile = {quantiles[0.25]:.0f}', 
            f'Median = {quantiles[0.5]:.0f}', 
            f'Upper quartile = {quantiles[0.75]:.0f}'] # f-Strings
)

# formatting
ax.minorticks_off()


##### Quantiles of random variables 

The definition of quantiles is essentially the same for random variables and observed data. In the case of random variables, one only needs to replace `cumulative relative frequencies` by `cumulative probabilities` (or, equivalently, by the `CDF`).

The `p`-quantile of a random variable is defined as the lowest value such that the cumulative probability associated with the value is at least `p`:

$$
Q_X(p)
=
\inf \{ r : p \leq \mathbb{P}[X \leq r] \}.
$$

### Introduction to some common theoretical distributions

The link between empirical distributions of observed variables and theoretical distributions of random variables is that of random sampling.

The thought experiment is that we suppose that the observed data are a random sample of independent realisations of a random variable with a given theoretical distribution.

Let us look at some common theoretical distributions of random variables.

Two important points to keep in mind:
1. `Observed data` are usually `messier` than what simple theoretical distributions would suggest.
2. `Observational equivalence`: a given sample of data can often be explained reasonably well by multiple candidate distributions.

#### Bernoulli distribution

This is the simplest probability distribution. It describes the outcome of a `binary` random variable, i.e. a random variable that can take on only two values.

By convention, these values are coded as:

 - `1`: a particular condition is satisfied / an event occurs / `True`
 - `0`: otherwise / `False`

Example:

For a dataset in which individuals are classified as either female or male, one can define the variable `female` as taking on the value `1` if the person is female and `0` otherwise.

In Economics, a `1`/`0`-coded binary variable is often called an `indicator variable` or `dummy variable`.

The Bernoulli distribution is completely characterised by a single parameter, usually denoted by `p`, which defines the probability of the random variable taking on the value `1`:

 - $\mathbb{P}[X = 1] = p$
 - $\mathbb{P}[X = 0] = 1 - p$



An example for a Bernoulli-distributed variable is the `holiday`-variable in the hotels-europe dataset. `holiday=1` indicates that the observation refers to a holiday and `holiday=0` indicates that the observation refers to day that is not a holiday.

In [ ]:
hotels_europe_restr_df['holiday'].value_counts(normalize=True).reset_index()

**Mean**

The mean of a Bernoulli-distributed random variable is simply equal to the probability of the variable taking on the value `1`.

The expected value is the probability-weighted sum of the possible values of the random variable:

$$
\mathbb{E}[X]
=
\mathbb{P}[X = 1]\cdot 1
+
\mathbb{P}[X = 0]\cdot 0
=
\mathbb{P}[X = 1]
=
p
$$

Analogously, the sample mean is the relative-frequency-weighted sum of the unique values observed in the sample:

$$
\bar{x}
=
\underbrace{\hat{p}[x = 1]}_{\mathclap{\substack{\text{relative}\\ \text{frequency of}\\ \text{\scriptsize x = 1}}}}
\cdot 1
+
\underbrace{\hat{p}[x = 0]}_{\mathclap{\substack{\text{relative}\\ \text{frequency of}\\ \text{\scriptsize x = 0}}}}
\cdot 0
=
\hat{p}[x = 1]
$$

Thus, the sample mean of a dummy variable is simply the relative frequency of the value `1`.

Let us check this equivalence for our variable `holiday`.

In [ ]:
df = hotels_europe_restr_df.copy()
print(len(df[df['holiday'] == 1]) / len(df[df['holiday'].notna()]))
print(df['holiday'].mean())

**Variance**

The variance of a Bernoulli-distributed random variable can be derived directly from the definition of variance as the expected value of squared deviations from the mean:

$$
Var[X]
=
\mathbb{E}[(X - \mathbb{E}[X])^2] 
=
\mathbb{P}[X = 1] \cdot (1 - \mathbb{E}[X])^2
+
\mathbb{P}[X = 0] \cdot (0 - \mathbb{E}[X])^2
$$

Since $\mathbb{E}[X] = p$, we obtain:

$$
p \cdot (1-p)^2
+
(1-p) \cdot p^2
$$

Factorising $p(1-p)$, we get:

$$
p \cdot (1-p) \cdot \big[(1-p)+p\big]
=
p \cdot (1-p)
$$

Thus,

$$
Var[X] = p \cdot (1-p)
$$

The variance is largest when $p=0.5$, in which case

$$
Var[X] = 0.25.
$$

The variance is equal to zero when $p=0$ or $p=1$, since the outcome is then known with certainty.

_Practical consequence_

Consider an opinion poll measuring support for a political party. All else equal, uncertainty is greatest when support is close to $50\%$ and smallest when support is close to $0\%$ or $100\%$. This is reflected in the fact that the variance $p(1-p)$ is maximised at $p=0.5$.

Thought experiment
p proportion of individuals support a certain party
we draw a random sample
try to estimate p
Var = p(1-p) --> maximum variance at p=0.5 (50%)
All else equal
The uncertainty of our estimation is the greatest when the true support for the party p is close to 50%

Let us quickly implement the formula for the variable `holiday` assuming our sample is the population.

In [ ]:
df = hotels_europe_restr_df.copy()
p = len(df[df['holiday'] == 1]) / len(df[df['holiday'].notna()])
print(p * (1-p))
print(df['holiday'].var(ddof=0))

discrete random variables (e.g. Bernoulli distribution)
--> probability mass function: each distinct value is associated with a particular probability of occurrence

continuous random variables (e.g. Normal distribution)
--> probability density function: each value is associated with a certain probability density of occurrence
--> probabilities are defined only for intervals of values
    --> area under probability density function (PDF)

#### Continuous probability distributions

There are variables that can in principle take on any numeric value within a given interval of real numbers. Typical examples are prices, incomes, waiting times, or distances.

Of course, observed data may exhibit heaping or bunching, meaning that values are rounded to particular multiples of integers. In many contexts, decimals may not even be important at all (e.g. annual income measured in euros).

Nevertheless, it is often convenient to model the underlying variable as if it could take on any value within a continuous interval.

For such variables, the probability of observing any exact value is effectively zero. Instead, probabilities are defined for intervals of values. To describe these probabilities, we use a `probability density function (PDF)`. The `probability` associated with a given interval corresponds to `the area under the density curve over that interval`.

At first glance, this may appear to complicate the exercise. In practice, however, continuous probability distributions often provide a simple description of otherwise complex data. Many widely used distributions can be characterised by only a small number of parameters.

As discussed previously, theoretical probability distributions are highly simplified representations of reality. Their purpose is not to perfectly reproduce every feature of an observed distribution, but rather to provide `an approximation` that captures its key characteristics.

##### Uniform distribution

Continuous Uniform distributions are the simplest continuous probability distributions.
1. Only values within a given interval may occur: between $x_{\min}$ and $x_{\max}$.
2. The probability density is the same for all values within this interval and zero elsewhere.

The probability density function (PDF) for a uniformly distributed variable on the interval $[x_{\min}, x_{\max}]$ is:

$$
f(x) =
\begin{cases}
\frac{1}{x_{\max}-x_{\min}} & \text{if } x \in [x_{\min}, x_{\max}] \\
0 & \text{otherwise}
\end{cases}
$$

Let us visualise two uniform PDFs.

In [ ]:
def uniform_pdf(x, xmin, xmax):
    return np.where(
        (x >= xmin) & (x <= xmax), # if-condition
        1 / (xmax - xmin), # if-condition satisfied
        0 # otherwise / else
    )

X = np.arange(-1, stop=2, step=0.01)
f_X_1 = uniform_pdf(x = X, xmin=0, xmax=1)
f_X_2 = uniform_pdf(x = X, xmin=-0.5, xmax=1.5)

fig, ax = plt.subplots()
ax.plot(X, f_X_1, **{'label': '$x_{\\min}=0$ and $x_{\\max}=1$'})
ax.plot(X, f_X_2, **{'label': '$x_{\\min}=-0.5$ and $x_{\\max}=1.5$'})
ax.set_ylabel('f(x)')
ax.set_xlabel('x')
ax.legend()

Let us focus on the uniform distribution with $x_{\min}=-0.5$ and $x_{\max}=1.5$. Let the random variable $X$ follow a Uniform distribution over the interval $[-0.5,1.5]$:

$$
X \sim \text{Unif}[-0.5,1.5]
$$

where the symbol $\sim$ should be read as "is distributed as" or "follows".

What is the probability that $X$ falls between $0.1$ and $0.5$? It corresponds to the area under the PDF over the interval $[0.1,0.5]$.

$$
\mathbb{P}[X\in [0.1, 0.5]] = \int_{0.1}^{0.5} \frac{1}{x_{\max} - x_{\min}}\ \text{d}x =  \int_{0.1}^{0.5} \frac{1}{1.5 - (-0.5)}\ \text{d}x = \int_{0.1}^{0.5} \frac{1}{2}\ \text{d}x 
$$

The antiderivative of a constant function is a linear function:

$$
\mathbb{P}[X\in [0.1, 0.5]] = \frac{1}{2} \cdot [x]_{0.1}^{0.5} = \frac{1}{2} \cdot [0.5 - 0.1] = \frac{0.4}{2}
$$

Therefore:

$$
\mathbb{P}[X\in [0.1, 0.5]] = 0.2 = 20\%
$$


In [ ]:
def uniform_pdf(x, xmin, xmax):
    return np.where(
        (x >= xmin) & (x <= xmax), # if-condition
        1 / (xmax - xmin), # if-condition satisfied
        0 # otherwise / else
    )

X = np.linspace(-1, 2, 300)
f_X_2 = uniform_pdf(x = X, xmin=-0.5, xmax=1.5)


fig, ax = plt.subplots()
ax.plot(X, f_X_2, label = 'Uniform PDF')
ax.fill_between(
    X,
    f_X_2,
    where=(X >= 0.1) & (X <= 0.5),
    alpha=0.3,
    **{'label': '$P(X \\in [0.1, 0.5])$'}
)

ax.set_ylabel('f(x)')
ax.set_xlabel('x')
ax.legend()

We can also use the cumulative distribution function to answer the previous question.

The cumulative distribution function of the uniform distribution is by definition the following:

$$
\mathbb{P}[X \leq v] \equiv F(v)  
= 
\int_{-\infty}^{v} f(s) \ \text{d}s 
= 
\begin{cases}
0 & \text{ if } v < x_{\min} \\ 
\int_{x_{\min}}^{v} \frac{1}{x_{\max} - x_{\min}} \ \text{d}s & \text{ if } v \in [x_{\min},x_{\max}] \\
1 & \text{ if } v > x_{\max} \\
\end{cases}
$$

Using the fact that the antiderivative of a constant function is a linear function:

$$
F(v) 
= 
\begin{cases}
0 & \text{ if } v < x_{\min} \\ 
\frac{v - x_{\min}}{x_{\max} - x_{\min}} & \text{ if } v \in [x_{\min},x_{\max}] \\
1 & \text{ if } v > x_{\max} \\
\end{cases}
$$

In the particular case, the values of the parameters are $x_{\min}=-0.5$ and $x_{\max}=1.5$:

$$
F(v) 
=
\begin{cases}
0 & \text{ if } v < -0.5 \\ 
\frac{v + 0.5}{2} & \text{ if } v \in [-0.5,1.5] \\
1 & \text{ if } v > 1.5 \\
\end{cases}
$$

Then the probability of $X$ taking on a value between $0.1$ and $0.5$ is:

$$

\mathbb{P}[X \in [0.1,0.5]] = \mathbb{P}[X \leq 0.5] - \mathbb{P}[X \leq 0.1] = F(0.5) - F(0.1) = \frac{0.5 + 0.5}{2} - \frac{0.1 + 0.5}{2} = \frac{0.4}{2} = 0.2 = 20\%

$$






In [ ]:
def uniform_cdf(x, xmin, xmax):
    return np.clip(
        (x - xmin) / (xmax - xmin),
        a_min=0,
        a_max=1
    )

xmin = -0.5
xmax = 1.5
xaxis_min = -1
xaxis_max = 2

X = np.arange(xaxis_min, stop=xaxis_max, step=0.01)
F_X_2 = uniform_cdf(x = X, xmin=xmin, xmax=xmax)

fig, ax = plt.subplots()
ax.plot(X, F_X_2, label = f'Uniform CDF with x_{\max} = ')

lb = 0.1
P_lb = uniform_cdf(x = lb, xmin=xmin, xmax=xmax)
ub = 0.5
P_ub = uniform_cdf(x = ub, xmin=xmin, xmax=xmax)

ax.yaxis.set_ticks(ticks=[0,P_lb,P_ub,1])
ax.xaxis.set_ticks(ticks=[xmin,lb,ub,xmax])

ax.plot([xaxis_min, lb], [P_lb, P_lb], linestyle='--', color='red')
ax.plot([xaxis_min, ub], [P_ub, P_ub], linestyle='--', color='red')
ax.plot([lb, lb], [0, P_lb], linestyle='--', color='red')
ax.plot([ub, ub], [0, P_ub], linestyle='--', color='red')



ax.set_ylabel('F(x)')
ax.set_xlabel('x')
ax.legend()

##### Simple logistic distribution

A commonly used family of bell-shaped symmetric distributions is the `logistic` family of probability distributions.

**CDF**

The simplest logistic distribution has the following `cumulative distribution function` (CDF):

$$
\mathbb{P}[X \leq v] \equiv F(v) = \frac{1}{1 + e^{-v}}
$$

Let us visualise this CDF.

In [ ]:
# defining function F(v) - formula above
#  
def simple_logistic_cdf(v):
    return 1 / (1 + np.exp(-v)) # np.exp() exponential from numpy

# computing the value of the function for a set of values between -8  and 8 
X = np.linspace(start=-8, stop=8, num=100) # alternative: np.arange 

# applying the function to the set of values X
F_X = simple_logistic_cdf(X)

fig, ax = plt.subplots(figsize=(6,6))
ax.plot(X, F_X, label='simple logistic distribution CDF')

ax.set_ylabel('F(x)')
ax.set_xlabel('x')
ax.legend()


**Quantile function**

Because the CDF has a particularly simple form, the corresponding quantile function is easy to derive.

Replacing $F(v)$ by $p$ (for the cumulative probability) and $v$ by $Q(p)$ (for the corresponding value):

$$
F(v) = \frac{1}{1 + e^{-v}} \ \ \Longleftrightarrow\ \ p = \frac{1}{1 + e^{-Q(p)}}
$$

Rearranging for $Q(p)$, one obtains the `quantile function`:

$$
Q(p) = \ln\left(\frac{p}{1-p}\right) 
$$

where $\frac{p}{1-p}$ is called the odds.

Let us visualise the quantile function.


In [ ]:
# defining the function Q(p)
def simple_logistic_quantile(p):
    return np.log(p/(1-p)) # np.log() --> natural logarithm from numpy

# cumulative probabilities: between 0 and 1
P = np.linspace(start=0.001, stop=0.999, num=1000)

# quantiles associated with the cumulative probabilities
# applying the quantile function Q() to the set of cumulative probabilities
Q_P = simple_logistic_quantile(P)

fig, ax = plt.subplots(figsize=(6,6))
ax.plot(P, Q_P, label='simple logistic distribution quantile function')

ax.set_ylabel('Q(p)')
ax.set_xlabel('p')
ax.legend()


continuous probability distributions
--> CDF (cumulative probability) -> PDF (probability density) 
   --> derivative
--> PDF (probability density) -> CDF (cumulative probability)
   --> integral


**PDF**

For continuous probability distributions:

- the cumulative distribution function (CDF) is the integral of the probability density function (PDF)
- the probability density function (PDF) is the derivative of the cumulative distribution function (CDF)

Using the latter, the `probability density function` of the simplest logistic probability distribution is:

$$
\text{probability density function }\equiv  f(v) = F'(v) = \frac{e^{-v}}{(1 + e^{-v})^2} = \frac{1}{(1 + e^{-v}) (1 + e^{v})}
$$

Let us visualise the probability density function to see its bell shape and symmetry.

In [ ]:
# defining the simple logistic PDF
def simple_logistic_pdf(v):
    return 1 / ((1 + np.exp(-v)) * (1 + np.exp(v)))

# set of values for the input
X = np.linspace(start=-8, stop=8, num=100)

# apply the function to the set of values
f_X = simple_logistic_pdf(X)

fig, ax = plt.subplots(figsize=(6,6))
ax.plot(X, f_X, label='simple logistic distribution PDF')

ax.set_ylabel('f(x)')
ax.set_xlabel('x')
ax.legend()

##### Exponential distribution

The family of exponential distributions is a simple asymmetric probability distribution, often used to model waiting times.

Intuition with waiting times:
> If an event occurs at a constant rate over time, then the exponential distribution describes the probability distribution of the waiting time until the event occurs for the first time.

This probability distribution is appropriate for variables that can only take non-negative values.

**PDF**

The probability density function of the exponential distribution is

$$
f(t) = 
\begin{cases}
\lambda \cdot e^{-\lambda \cdot t} & \text{ if } t \geq 0 \\
0 & \text{otherwise}
\end{cases}
$$

where $t$ is the waiting time until the first occurrence of the event and $\lambda$ is the rate parameter. The higher $\lambda$, the more likely the event is to occur in a given period of time.

The expected value of the exponentially-distributed random variable $T$ is 

$$
\mathbb{E}[T] = \frac{1}{\lambda}
$$

This corresponds to the average waiting time until the event occurs.

Let us visualise the PDF of the exponential distribution.


In [ ]:

# defining the PDF for the exponential distribution
# for each value of waiting time t and rate of occurrence rate_paramter (lambda) 
def exponential_pdf(t, rate_parameter):
    return np.where(t >= 0, # only positive values --> waiting times can only be non-negative
                    rate_parameter * np.exp(-rate_parameter*t), # formula
                    0) # np.where() sort of if-then-else  

# set of values for waiting time
T = np.linspace(start=0, stop=10, num=100)

fig, ax = plt.subplots(figsize=(6,6))
# PDF for three values of the rate parameter lambda
ax.plot(T, exponential_pdf(t=T, rate_parameter=0.5), label=r'$\lambda = 0.5$')
ax.plot(T, exponential_pdf(t=T, rate_parameter=1.0), label=r'$\lambda = 1.0$')
ax.plot(T, exponential_pdf(t=T, rate_parameter=2.0), label=r'$\lambda = 2.0$')

ax.set_ylabel('f(t)')
ax.set_xlabel('t')
ax.legend()

PDF --> CDF
integral
CDF --> Quantile
inverse

**CDF and quantile function**

Under the waiting time interpretation, the cumulative distribution function (CDF) of an exponentially distributed random variable determines for each value $t$ the probability that we need to wait at most $t$:

$$
\mathbb{P}[T \leq t] \equiv F(t)
$$

By definition, the CDF is the integral of the PDF, such that:

$$
F(t) = \int_{0}^{t} f(v)\ \text{d}v = \int_{0}^{t} \lambda \cdot e^{-\lambda \cdot v}\ \text{d}v
$$

Using the fact that the antiderivative of $ g(v)=(-\lambda)\cdot e^{-\lambda\cdot v} $ is $ e^{-\lambda\cdot v}$, we obtain:

$$
F(t) 
= (-1)\cdot \int_{0}^{t} (-\lambda)\cdot e^{-\lambda\cdot v} \ \text{d}v 
= (-1)\cdot \left[ e^{-\lambda\cdot v} \right]_{v=0}^{v=t}
= e^{-\lambda\cdot 0} - e^{-\lambda\cdot t}
$$

Thus, the CDF corresponds to

$$
F(t) = 1 - e^{-\lambda\cdot t}
$$

This result is intuitive: the larger $t$, the higher the probability that the waiting time until the first occurrence of the event is at most $t$.

To derive the quantile function, we compute the inverse of the CDF:

$$
p = 1 - e^{-\lambda\cdot Q_T(p)} \ \ \Longleftrightarrow \ \ Q_T(p) = \frac{\ln\left(\frac{1}{1-p}\right)}{\lambda}
$$

**Mean–median comparison**

The `mean` waiting time is

$$
\mathbb{E}[T] = \frac{1}{\lambda}
$$

The `median` waiting time is

$$
Q_T(0.5) = \frac{\ln(2)}{\lambda} = \ln(2) \cdot \mathbb{E}[T] \approx 0.69 \cdot \mathbb{E}[T]
$$

Since $\ln(2) < 1$ and $\lambda > 0$, the median is smaller than the mean:

$$
Q_T(0.5) < \mathbb{E}[T].
$$

This is not surprising because the exponential distribution is `right-skewed`. For right-skewed distributions, `the mean lies above the median`.

Let us visualise the PDF, the mean, and the median for $\lambda=1$.

In [ ]:
# defining the Exponential-PDF
def exponential_pdf(t, rate_parameter):
    return np.where(t >= 0,
                    rate_parameter * np.exp(-rate_parameter*t),
                    0)

# choosing lambda
rate_parameter_value = 10
# mean (applying the formula)
expected_value = 1/rate_parameter_value
# median (applying the formula)
median = np.log(2)/rate_parameter_value 

# set of values of waiting time for the PDF
T = np.linspace(start=0, stop=expected_value * 2.5, num=10000)
# applying PDF to the set of values
f_T = exponential_pdf(t=T, rate_parameter=rate_parameter_value)

fig, ax = plt.subplots(figsize=(10,3))

ax.plot(T, f_T, label=rf'PDF ($\lambda = {rate_parameter_value:.1f}$)')

#highlight the area under the PDF below the median
ax.fill_between(
    T,
    f_T,
    where=(T >= 0) & (T <= median),
    alpha=0.3,
    **{'label': rf'Area = 0.5 ($50\%$)'}
) 

#labelling the axes
ax.xaxis.set_ticks(ticks=[0, median, expected_value], labels=['0', f'median\n{median:.2f}', f'mean\n{expected_value:.2f}'])

# extra lines at the median and mean
ax.axvline(
    expected_value,
    linestyle='--',
    color='red',
    label='mean',
    linewidth = 3
)

ax.axvline(
    median,
    linestyle='--',
    color = 'green',
    label='median',
    linewidth = 3
)


ax.set_ylim(bottom=0)
ax.set_ylabel('f(t)')
ax.set_xlabel('t')
ax.legend()

##### Normal distribution

A distribution that frequently appears in statistics, particularly in statistical inference, is the family of normal distributions.

Like the logistic distribution, the normal distribution is symmetric and bell-shaped. (More on this below.)

Each normal distribution has two parameters: the mean $\mu$ and the variance $\sigma^2$.

Suppose that the random variable $X$ is normally distributed with mean $\mu$ and variance $\sigma^2$. We write this as:

$$
X \sim \mathcal{N}(\mu, \sigma^2)
$$

Let $Z$ be the random variable, obtained by standardising $X$ (i.e. by subtracting the mean $\mu$ and dividing by the standard deviation $\sigma$):

$$
Z = \frac{X - \mu}{\sigma}
$$

The standardised random variable $Z$ is then standard-normally distributed:

$$
Z \sim \mathcal{N}(0, 1)
$$

The `probability density function (PDF)` of the standard normal distribution corresponds to:

$$
\phi(z) = \frac{1}{\sqrt{2\pi}} \cdot e^{-z^2/2}
$$

Personal remark: For its significance in statistics, it is a relatively simple and elegant formula. It contains the constants $\pi$ and $e$, as well as a quadratic. 

Let us plot the probability density function of the normal distribution.



In [ ]:
# parameters
mu = 3 # mean
sigma = 5 # standard deviation

def phi(z):  # standard normal PDF - formula above
    return (1 / np.sqrt(2 * np.pi)) * np.exp(-(z**2) / 2)


def normal_pdf(x, mu, sigma): # normal distribution with mean mu and st dev sigma
    z = (x - mu) / sigma
    return phi(z) / sigma

# limits of domain for the plot 
xaxis_min = mu - 3.5 * sigma # lower bound
xaxis_max = mu + 3.5 * sigma # upper bound

# set of values for X
X = np.linspace(start=xaxis_min, stop=xaxis_max, num=10000)
# apply the pdf to the set of values
f_X = normal_pdf(x=X, mu=mu, sigma=sigma)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(X, f_X)


In [ ]:
from scipy.stats import norm

mu = 0
sigma = 1

def phi(z):  # standard normal PDF
    return (1 / np.sqrt(2 * np.pi)) * np.exp(-(z**2) / 2)

def normal_pdf(x, mu, sigma):
    z = (x - mu) / sigma
    return phi(z) / sigma

xaxis_min = mu - 3.5 * sigma
xaxis_max = mu + 3.5 * sigma

X = np.linspace(start=xaxis_min, stop=xaxis_max, num=10000)
f_X = normal_pdf(x=X, mu=mu, sigma=sigma)

fig, axs = plt.subplots(
    nrows=3,
    ncols=1,
    sharex=False,
    sharey=True,
    figsize=(6, 10)
)

ticks = [
    mu - 3*sigma,
    mu - 2*sigma,
    mu - sigma,
    mu,
    mu + sigma,
    mu + 2*sigma,
    mu + 3*sigma
]

tick_labels = [
    rf'$\mu - 3\sigma$' + f'\n{mu - 3*sigma:.2f}',
    rf'$\mu - 2\sigma$' + f'\n{mu - 2*sigma:.2f}',
    rf'$\mu - \sigma$' + f'\n{mu - sigma:.2f}',
    rf'$\mu$' + f'\n{mu:.2f}',
    rf'$\mu + \sigma$' + f'\n{mu + sigma:.2f}',
    rf'$\mu + 2\sigma$' + f'\n{mu + 2*sigma:.2f}',
    rf'$\mu + 3\sigma$' + f'\n{mu + 3*sigma:.2f}'
]

area_1sd = norm.cdf(mu + sigma, loc=mu, scale=sigma) - norm.cdf(mu - sigma, loc=mu, scale=sigma)
area_2sd = norm.cdf(mu + 2*sigma, loc=mu, scale=sigma) - norm.cdf(mu - 2*sigma, loc=mu, scale=sigma)
area_3sd = norm.cdf(mu + 3*sigma, loc=mu, scale=sigma) - norm.cdf(mu - 3*sigma, loc=mu, scale=sigma)

for ax in axs:
    ax.plot(X, f_X, label=rf'PDF ($\mu = {mu:.1f}$, $\sigma = {sigma:.1f}$)')
    ax.set_xlim(left=xaxis_min, right=xaxis_max)
    ax.set_ylim(bottom=0)
    ax.set_xticks(ticks, labels=tick_labels)
    ax.set_ylabel('f(x)')
    ax.set_xlabel('x')

axs[0].fill_between(
    X,
    f_X,
    where=(X >= mu - sigma) & (X <= mu + sigma),
    alpha=0.3,
    label=rf'Area = {area_1sd * 100:.1f}%'
)

axs[1].fill_between(
    X,
    f_X,
    where=(X >= mu - 2*sigma) & (X <= mu + 2*sigma),
    alpha=0.3,
    label=rf'Area = {area_2sd * 100:.1f}%'
)

axs[2].fill_between(
    X,
    f_X,
    where=(X >= mu - 3*sigma) & (X <= mu + 3*sigma),
    alpha=0.3,
    label=rf'Area = {area_3sd * 100:.1f}%'
)


axs[0].axvline(mu - sigma, linestyle='--', linewidth=3)
axs[0].axvline(mu + sigma, linestyle='--', linewidth=3)

axs[1].axvline(mu - 2*sigma, linestyle='--', linewidth=3)
axs[1].axvline(mu + 2*sigma, linestyle='--', linewidth=3)

axs[2].axvline(mu - 3*sigma, linestyle='--', linewidth=3)
axs[2].axvline(mu + 3*sigma, linestyle='--', linewidth=3)


axs[0].legend()
axs[1].legend()
axs[2].legend()

##### Comparison Normal distribution vs. Logistic distribution with same mean and variance

The normal distribution and the logistic distribution are both families of bell-shaped symmetric probability distributions. They are often used in statistics, econometrics, and data science. One example is binary response models, where the probit model is based on the normal distribution and the logit model on the logistic distribution.

Let us compare the probability density functions of the two distributions when they have the same mean and variance.

In [ ]:
mu = 0
sigma = 1

def phi(z):  # standard normal PDF
    return (1 / np.sqrt(2 * np.pi)) * np.exp(-(z**2) / 2)

def normal_pdf(x, mu, sigma): # normal PDF
    z = (x - mu) / sigma
    return phi(z) / sigma

def simple_logistic_pdf(v): # simple logistic PDF
    return 1 / ((1 + np.exp(-v)) * (1 + np.exp(v)))

def logistic_scale(sigma): # logistic scale as a function of standard deviation
    return np.sqrt(3) * sigma / np.pi

def logistic_pdf(x, mu, scale): # logistic pdf
    v = (x - mu) / scale
    return simple_logistic_pdf(v) / scale

xaxis_min = mu-5*sigma
xaxis_max = mu+5*sigma

X = np.linspace(start=xaxis_min, stop= xaxis_max, num=1000)
f_normal = normal_pdf(X, mu=mu, sigma=sigma)
f_logistic = logistic_pdf(X, mu=mu, scale=logistic_scale(sigma))
fig, ax = plt.subplots(figsize=(6,4))
ax.plot(X, f_normal, color='blue', linestyle='-', label='Normal PDF')
ax.plot(X, f_logistic, color='red', linestyle='--', label='Logistic PDF', linewidth=3)


ticks = [
    mu - 3*sigma,
    mu - 2*sigma,
    mu - sigma,
    mu,
    mu + sigma,
    mu + 2*sigma,
    mu + 3*sigma
]

tick_labels = [
    rf'$\mu - 3\sigma$' + f'\n{mu - 3*sigma:.2f}',
    rf'$\mu - 2\sigma$' + f'\n{mu - 2*sigma:.2f}',
    rf'$\mu - \sigma$' + f'\n{mu - sigma:.2f}',
    rf'$\mu$' + f'\n{mu:.2f}',
    rf'$\mu + \sigma$' + f'\n{mu + sigma:.2f}',
    rf'$\mu + 2\sigma$' + f'\n{mu + 2*sigma:.2f}',
    rf'$\mu + 3\sigma$' + f'\n{mu + 3*sigma:.2f}'
]

ax.set_xlim(left=xaxis_min, right=xaxis_max)
ax.set_ylim(bottom=0)
ax.set_xticks(ticks, labels=tick_labels)
ax.set_ylabel('f(x)')
ax.set_xlabel('x')

ax.legend()


The two distributions have the same mean $\mu$ and variance $\sigma^2$. The logistic distribution is slightly more concentrated around the centre and has slightly heavier tails than the normal distribution. (In technical terms, the logistic distribution has higher excess kurtosis than the normal distribution.)

##### Illustration of the Central Limit Theorem

The Central Limit Theorem (CLT) is a key result in inferential statistics involving the Normal distribution.

For samples consisting of `independent draws` from a random variable with `finite mean and variance`:

`as the sample size increases, the distribution of the sample mean more and more closely resembles a Normal distribution`.

Let us illustrate the CLT using repeated random samples from an imaginary large population of one million individuals with known proportions of supporters of three political parties:

 - 450 thousand supporters of the Idealist Party
 - 450 thousand supporters of the Realist Party 
 - 100 thousand supporters of the Self-Critical Party


In [ ]:
rng = np.random.default_rng(39258)

preference = (
    ['Realist Party'] * 450_000
    + ['Idealist Party'] * 450_000
    + ['Self-Critical Party'] * 100_000
)
rng.shuffle(preference)
population_df = pd.DataFrame({'person_id': np.arange(stop=1000_000, step=1) + 1,
                              'preferred_party': pd.Categorical(preference)})

population_df

Preferences in the population:

In [ ]:
population_df['preferred_party'].value_counts(normalize=True).reset_index()

Let us draw a random sample a random sample without replacement and diplay the preferences in the sample:

In [ ]:
rng = np.random.default_rng(40932)

random_person_ids = rng.choice(
    population_df['person_id'],
    replace=False,
    size=1000
)

random_sample = population_df.loc[population_df['person_id'].isin(random_person_ids),:].copy()

random_sample['preferred_party'].value_counts(normalize=True).reset_index()


Let us draw another random sample and display the preferences.

In [ ]:
rng = np.random.default_rng(75189)

random_person_ids = rng.choice(
    population_df['person_id'],
    replace=False,
    size=1000
)

random_sample = population_df.loc[population_df['person_id'].isin(random_person_ids),:].copy()

random_sample['preferred_party'].value_counts(normalize=True).reset_index()

We can see that proportions vary from sample to sample.

Therefore, we can define a distribution of the sample average support for the Idealist Party.

Let us define the binary variable $D$ as an indicator for whether a person supports the Idealist Party:

$$
D =
\begin{cases}
1 & \text{ if the person supports the Idealist Party} \\
0 & \text{ otherwise}
\end{cases}
$$

Then $D$ is a Bernoulli-distributed random variable.

The population mean of $D$ is equal to the proportion of supporters of the Idealist Party in the population:

$$
\mathbb{E}[D] = 0.45
$$

This value is assumed to be fixed.

The sample mean of $D$ is equal to the proportion of supporters of the Idealist Party in sample $S$:

$$
\bar{D}(S) = \frac{1}{n} \sum_{i \in S} D_i
$$

The CLT says that, as the sample size $n$ increases, the distribution of the sample mean $\bar{D}(S)$ more and more closely resembles a Normal distribution.

Let us draw $R = 100000$ random samples to illustrate this.

In [ ]:
population_df['D'] = np.where(population_df['preferred_party'].eq('Idealist Party'),
                              1,
                              0)

np.random.default_rng(42691)

D = population_df['D'].astype(int).to_numpy()
R = 100000
sample_means = np.empty(R)

for r in range(R):
    sample_means[r] = rng.choice(D, size=1000, replace=False).mean()

random_samples_df = pd.DataFrame({
    'sample_id': np.arange(1, R+1),
    'sample_mean': sample_means
})

Let us visualise the distribution of sample means (mean support for the Idealist Party):

In [ ]:
df2 = random_samples_df.copy()


# creating bins of sample mean values, intervals for the values of the sample mean 
bins = np.arange(
    df2['sample_mean'].min(),
    df2['sample_mean'].max() + 0.001,
    0.001
)
df2['sample_mean_bin'] = pd.cut(df2['sample_mean'], bins=bins)


# relative frequencies for each interval / bin of sample mean value
df2 = (
    df2['sample_mean_bin']
    .value_counts(normalize=True, sort=False)
    .reset_index()
)

df2.columns = ['sample_mean_bin', 'proportion']

df2 = df2.loc[df2['proportion'] > 0, :].copy()

# taking the midpoint of the interval to the x / horizontal axis
df2['sample_mean'] = df2['sample_mean_bin'].astype(object).map(lambda x: x.mid)

fig, ax = plt.subplots(figsize=(6, 6))

ax.bar(
    x=df2['sample_mean'],
    height=df2['proportion'],
    width=0.01
)



Let us produce a percentile-percentile plot to show that the distribution of sample means indeed looks like a normal distribution.

In [ ]:
# interested in percentiles, so values Q(p) for p=0.01, 0.02, 0.03, ..., 0.99
cumulative_proportions = np.arange(start=0.01, step=0.01, stop=1)

# empirical percentiles
percentiles_of_sample_means = [random_samples_df['sample_mean'].quantile(p) for p in cumulative_proportions]


# mean and st dev to define the benchmark / point of comparison normal distribution
mean_of_sample_means = random_samples_df['sample_mean'].mean()
stdev_of_sample_means = random_samples_df['sample_mean'].std(ddof=0)

# normal distribution percentiles
from scipy import stats
q_normal = stats.norm.ppf(cumulative_proportions, loc=mean_of_sample_means, scale=stdev_of_sample_means) 

In [ ]:


fig, ax = plt.subplots(figsize=(6,6))
# x / horizontal axis -> normal-distribution percentiles
# y / vertical axis -> observed empirical percentiles 
ax.scatter(x=q_normal, y=percentiles_of_sample_means)

# add a 45° line
ax.plot(
    [q_normal.min(), q_normal.max()],
    [q_normal.min(), q_normal.max()]
) 

ax.set_xlabel('Normal distribution percentiles')
ax.set_ylabel('Percentiles of sample means')

#### Distribution of labor income in the NLSY97 sample 

Let us use data from the NLSY 1997 cohort on yearly labor income (from wages and salaries) `yinc` and yearly hours of work `yhours`.

Let us use this time only yearly income from year 2023. Members of the cohort were born between 1980 and 1984, this means that they were about fourty years old in 2023.  

In [ ]:
df3 = nlsy97_income_hours_df.copy()
# year 2023
# yinc yearly income is not missing
# focus on yinc yearly income
df3 = df3.loc[(df3['year'] == 2023) & df3['yinc'].notna(), ['yinc']].reset_index(drop=True)

In [ ]:
df3.shape

As usual, let us begin with a histogram of relative frequencies.

In [ ]:
df4 = df3.copy()
df4 = (
    df4
        .value_counts(subset='yinc', normalize=True) # relative frequencies for each value of income
        .reset_index() # formatting to have two columns: yinc, proportion
        .sort_values(by='yinc', ascending=True) # sorting the values by income
        .reset_index(drop=True) # formatting
)
df4


In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(x=df4['yinc'], height=df4['proportion'], width=1000)
ax.set_xlabel('Yearly Income')
ax.set_ylabel('Relative frequency')


Because of the large mass point at `yinc = 416374`, the data appear to be top-coded (or right-censored).

There also appears to be substantial heaping (round-number bunching) at multiples of 1,000.

Let us therefore redraw the plot using 5,000-dollar income bins.

In [ ]:
df4 = df3.copy()
# bin observations
# --> group observations together by intervals of income
# --> compute relative frequencies for each interval
# pd.cut() 
# defining the intervals 
# [0,5000),  [5000,10000) etc
# start from min
# stop a bit above the maximum value
limit_points_intervals = np.arange(start=df4['yinc'].min(), 
                                   step=5000, 
                                   stop=(df4['yinc'].max() + 5000))


df4['yinc_bin'] = pd.cut(df4['yinc'], bins=limit_points_intervals)

# computing relative frequencies for the intervals
df4 = (
    df4
        .value_counts(subset='yinc_bin', normalize=True) # relative frequencies
        .reset_index() # formatting
        .sort_values(by='yinc_bin', ascending=True) # sorting
        .reset_index(drop=True) # formatting
)

# plotting intervals is difficult because there two values for each interval
# take the midpoint of each interval 
df4['yinc_bin_midpoint'] = df4['yinc_bin'].cat.categories.mid
# Maybe another possible way:
# df4['yinc_bin_midpoint'] = df4['yinc_bin'].map(lambda x: x.mid)


In [ ]:
fig, ax = plt.subplots(figsize=(8,4))

# bar plot
# x axis midpoint of each interval
# y axis relative frequencies for the intervals 
# width of the bars to be adjusted manually according to the units
ax.bar(x=df4['yinc_bin_midpoint'], height=df4['proportion'], width=4000)
ax.set_xlabel('Yearly Income')
ax.set_ylabel('Relative frequency')

Right-skewed distribution --> asymmetric with long right tail
median < mean 

One candidate for income distributions is the log-normal distribution
`income` is `log-normally` distributed **if** `ln(income)` is `normally` distributed


#### Checking whether the data appear to follow a log-normal distribution

Suppose that the observations were drawn independently from a common continuous distribution. In other words, suppose that the observed income values are independent realisations of a random variable $Y$.

A common model for income is the log-normal distribution.

What does this mean?

A variable $Y$ is said to be log-normally distributed if its natural logarithm, $\ln(Y)$, is normally distributed.

Therefore, one way to assess whether income may be approximately log-normally distributed is to examine the distribution of $\ln(\text{income})$.

If the logarithm of income looks approximately normally distributed, then the log-normal distribution may provide a reasonable description of the data.

Let us investigate whether this appears to be the case.

In [ ]:
df5 = df3.copy()
# ln(0) is not defined
# one possibility --> add a small value to income for all observations
# another possibility --> drop observations with 0 income
# let us drop the 0-income observations
df5 = df5.loc[df5['yinc'] > 0, :].reset_index(drop=True)

# taking the natural logarithm
df5['ln_yinc'] = np.log(df5['yinc']) #numpy function np.log

df6 = df5.copy()

# computing relative frequencies for values of log income
df6 = (
    df6
        .value_counts(subset='ln_yinc', normalize=True) # relative frequencies
        .reset_index() # formatting
        .sort_values(by='ln_yinc', ascending=True) # sorting
        .reset_index(drop=True) # formatting
        .copy()
)
df6

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(x=df6['ln_yinc'], height=df6['proportion'], width=0.1)
ax.set_xlabel('Ln(Yearly Income)')
ax.set_ylabel('Relative frequency')

Let us bin once again the observations in intervals and produce the histogram.

In [ ]:
df7 = df5.copy()
# binning observations
# create intervals of income with pd.cut()

interval_bounds = np.arange(start=df7['ln_yinc'].min(), # minimum
                            step=0.2, # interval width
                            stop=(df7['ln_yinc'].max() + 2) # little above maximum 
                            )

df7['ln_yinc_bin'] = pd.cut(df7['ln_yinc'], bins=interval_bounds)

# relative frequencies for the binned variable
df7 = (
    df7
        .value_counts(subset='ln_yinc_bin', normalize=True)
        .reset_index()
        .sort_values(by='ln_yinc_bin', ascending=True)
        .reset_index(drop=True)
)


# for plotting --> define midpoint for each interval
df7['ln_yinc_bin_midpoint'] = df7['ln_yinc_bin'].map(lambda x: x.mid)

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(x=df7['ln_yinc_bin_midpoint'], height=df7['proportion'], width=0.1)
ax.set_xlabel('Ln(Yearly Income)')
ax.set_ylabel('Relative frequency')

The histogram suggests that log income is somewhat more symmetric and bell-shaped than income itself, although some asymmetry is also visible. Before we had a right-skewed distribution. Now it looks a little left-skewed.

However, visual impressions from histograms can be misleading and depend on the choice of bins.

Let us consider a quantile-quantile (Q-Q) plot.

A Q-Q plot compares the empirical quantiles of the data with the quantiles implied by a theoretical distribution.

If log income were exactly normally distributed, the points would lie approximately on a straight line.

Systematic deviations from a straight line indicate departures from normal distribution.

In the case of the CLT we produced a Percentile-Percentile plot
 -> which is a version of a Quantile-Quantile plot

Let us make a Quantile-Quantile plot
 -> plotting empirical quantiles of ln(income)
 -> against the corresponding normal distribution quantiles

To see how far we are from a normal distribution

Preparing empirical quantiles and summary stats.

In [ ]:
# mean and st dev are needed for the benchmark normal distribution
mean = df5['ln_yinc'].mean()
sd = df5['ln_yinc'].std()

# original data 
empirical_quantiles = (
    df5['ln_yinc']
        .sort_values()
        .reset_index(drop=True)
        .copy()
)
empirical_quantiles
# every observation can be thought of as a quantile
 # value of the variable Q(p) for the given cumulative proportion p
# the corresponding cumulative proportion p  
 # can be inferred from the rank of the observation 
 # when the observations are arranged in ascending order   
# arrange observations in ascending order of ln(income)
# every observation has a rank
# rank divided by number of observations 
  # -> approximate cumulative proportion p corresponding to that value / observation Q(p)

# need the number of observations:
n = len(empirical_quantiles) # number of empirical quantiles


Calculating theoretical quantiles.

In [ ]:

# cumulative proportions to be used for the normal quantiles
# approximate cumulative proportions/probabilities associated with the ordered observations
# instead of this 1/n, 2/n, 3/n, n/n etc 
# let us have 0.5/n, 1.5/n, ... , (n-0.5)/n
p = (np.arange(1, n+1) - 0.5)/n 

# normal quantiles associated with the cumulative proportions
from scipy import stats
# theoretical quantiles associated with the cumulative proportions/probabilities
q_normal = stats.norm.ppf(p, loc=mean, scale=sd) 


# compare empirical_quantiles with q_normal for the same values of cumulative proportions p

Plotting.

In [ ]:
fig, ax = plt.subplots(figsize=(6,6))
ax.scatter(x=q_normal, y=empirical_quantiles)
ax.plot(
    [q_normal.min(), q_normal.max()],
    [q_normal.min(), q_normal.max()]
) # 45° line
ax.set_xlabel('Theoretical normal distribution quantile')
ax.set_ylabel('Empirical quantile of Ln(Yearly Income)')

#### Exercise

Let us repeat the previous analysis for hourly income, defined as

$$
hinc = \frac{yinc}{yhours}.
$$

Specifically:

1. Create the variable `hinc`.
2. Plot the distribution of `hinc`.
3. Plot the distribution of `ln(hinc)`.
4. Create a Q-Q plot for `ln(hinc)` to compare it with a normal distribution.


To avoid extreme values due to very low annual hours, restrict the sample to observations with `yhours > 500`.